# 13 — Factless Fact Table — Spark SQL

Grain: `(ActivityDateKey, EmployeeSK, TerritorySK)` — evento sem métricas.

**Técnica Spark SQL:** `INSERT INTO ... SELECT DISTINCT` via JOINs.

In [1]:
import sys
import os
sys.path.insert(0, os.getcwd())
from utils import get_spark, register_catalog, WAREHOUSE_DIR

spark = get_spark("NorthwindDW SQL - 13 Factless Fact")
print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/29 01:01:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.0


In [2]:
register_catalog(spark)

Catálogo registrado: {'bronze': 11, 'silver': 0, 'gold': 14}


In [3]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS gold.FactEmployeeTerritoryActivity (
        ActivityDateKey INT, EmployeeSK INT, TerritorySK INT
    ) USING DELTA
""")

spark.sql("DELETE FROM gold.FactEmployeeTerritoryActivity")
spark.sql("""
    INSERT INTO gold.FactEmployeeTerritoryActivity
    SELECT DISTINCT
        CAST(DATE_FORMAT(CAST(o.OrderDate AS DATE), 'yyyyMMdd') AS INT) AS ActivityDateKey,
        de.EmployeeSK,
        b.TerritorySK
    FROM bronze.orders o
    JOIN gold.DimEmployee de ON o.EmployeeID = de.EmployeeID
    JOIN gold.BridgeEmployeeTerritory b ON b.EmployeeSK = de.EmployeeSK
""")

n = spark.sql("SELECT COUNT(*) AS n FROM gold.FactEmployeeTerritoryActivity").collect()[0]["n"]
print(f"FactEmployeeTerritoryActivity: {n} eventos")
assert n > 0
print("Factless fact carregada OK")

FactEmployeeTerritoryActivity: 3696 eventos
Factless fact carregada OK


In [4]:
spark.sql("""
    SELECT COUNT(DISTINCT TerritorySK) AS TerritoriosCobertos,
           (SELECT COUNT(*) FROM gold.DimTerritory) AS TerritoriostTotal
    FROM gold.FactEmployeeTerritoryActivity
""").show()

+-------------------+-----------------+
|TerritoriosCobertos|TerritoriostTotal|
+-------------------+-----------------+
|                 49|               53|
+-------------------+-----------------+



In [5]:
spark.sql("""
    SELECT d.Year, dt.RegionName,
           COUNT(DISTINCT fa.TerritorySK) AS TerritoriosCobertos,
           COUNT(DISTINCT fa.EmployeeSK)  AS EmpregadosAtivos,
           COUNT(*) AS TotalEventos
    FROM gold.FactEmployeeTerritoryActivity fa
    JOIN gold.DimDate d ON d.DateKey = fa.ActivityDateKey
    JOIN gold.DimTerritory dt ON dt.TerritorySK = fa.TerritorySK
    GROUP BY d.Year, dt.RegionName ORDER BY d.Year, dt.RegionName
""").show(20)

+----+--------------------+-------------------+----------------+------------+
|Year|          RegionName|TerritoriosCobertos|EmpregadosAtivos|TotalEventos|
+----+--------------------+-------------------+----------------+------------+
|1996|Eastern          ...|                 19|               4|         327|
|1996|Northern         ...|                 11|               2|         111|
|1996|Southern         ...|                  4|               1|          72|
|1996|Western          ...|                 15|               2|         185|
|1997|Eastern          ...|                 19|               4|         734|
|1997|Northern         ...|                 11|               2|         341|
|1997|Southern         ...|                  4|               1|         264|
|1997|Western          ...|                 15|               2|         500|
|1998|Eastern          ...|                 19|               4|         494|
|1998|Northern         ...|                 11|               2|

In [6]:
spark.sql("""
    SELECT dt.RegionName, dt.TerritoryDescription, de.FullName AS EmpregadoResponsavel
    FROM gold.BridgeEmployeeTerritory b
    JOIN gold.DimTerritory dt ON dt.TerritorySK = b.TerritorySK
    JOIN gold.DimEmployee de ON de.EmployeeSK = b.EmployeeSK
    WHERE NOT EXISTS (
        SELECT 1 FROM gold.FactEmployeeTerritoryActivity fa
        WHERE fa.EmployeeSK = b.EmployeeSK AND fa.TerritorySK = b.TerritorySK
    )
    ORDER BY dt.RegionName
""").show(truncate=False)

+----------+--------------------+--------------------+
|RegionName|TerritoryDescription|EmpregadoResponsavel|
+----------+--------------------+--------------------+
+----------+--------------------+--------------------+

